# MTG Metagame GNN - Training Notebook

Train the Heterogeneous Graph Transformer (HGT) for MTG Standard metagame prediction on Google Colab with GPU acceleration.

**Runtime**: Select `Runtime > Change runtime type > T4 GPU` before running.

## 1. Setup

In [ ]:
# Mount Google Drive (stores data and model checkpoints)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo (or pull latest)
import os

REPO_DIR = '/content/mtg-graph'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/crosis19/mtg-graph.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install dependencies
!pip install -q -e .
!pip install -q pyarrow

In [ ]:
# Symlink data from Google Drive (persistent storage)
# Adjust this path to match your Drive folder structure
DRIVE_DATA = '/content/drive/MyDrive/mtg-graph/data'

if os.path.exists(DRIVE_DATA):
    # Use existing data from Drive
    if not os.path.islink('data'):
        !rm -rf data
        !ln -s {DRIVE_DATA} data
    print('Linked data from Google Drive')
else:
    # Create data directories
    os.makedirs('data/raw', exist_ok=True)
    os.makedirs('data/processed', exist_ok=True)
    os.makedirs('data/embeddings', exist_ok=True)
    print('Created fresh data directories (will need to run ingestion)')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Data Ingestion (Optional)

Skip this section if you already have processed data in Google Drive.

In [ ]:
# Option A: Generate synthetic data (fast, for testing)
# !python -m src.synthetic_data

# Option B: Run real data ingestion (slow, requires internet)
# !python -m src.ingest_cards
# !python -m src.ingest_tournaments
# !python -m src.ingest_metagame
# !python -m src.ingest_matchups

## 3. Build Synergy Edges & Graph

In [ ]:
# Build synergy edges (Phase 2)
# Only needed if you ran data ingestion above
# !python -m src.keyword_matrix
# !python -m src.card_embeddings

# Build heterogeneous graph (Phase 3)
!python -m src.graph_builder

## 4. Train Model (GPU-Accelerated)

In [ ]:
import logging
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.data import HeteroData

from src.config import (
    DECKLISTS_PARQUET, DROPOUT, EMERGENCE_LOSS_WEIGHT, GRAPH_PATH,
    HIDDEN_DIM, LEARNING_RATE, METAGAME_PARQUET, MODEL_PATH,
    NUM_EPOCHS, NUM_HEADS, NUM_HGT_LAYERS, TOP8_LOSS_WEIGHT,
    TOURNAMENTS_PARQUET, TRAIN_SPLIT_RATIO, VAL_SPLIT_RATIO, WEIGHT_DECAY,
)
from src.model import MTGMetagameHGT
from src.train import (
    _build_emergence_targets, _build_top8_samples, _temporal_split, _evaluate_top8,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Seed for reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

# Load graph
print('Loading graph...')
data = torch.load(GRAPH_PATH, weights_only=False)
print(f'  Nodes: {{", ".join(f"{nt}={data[nt].x.shape[0]}" for nt in data.node_types)}}')
print(f'  Edge types: {len(data.edge_types)}')

# Build targets
print('\nBuilding targets...')
emergence_targets = _build_emergence_targets(data)
arch_idx, tourney_idx, top8_labels = _build_top8_samples(data)
print(f'  Emergence targets: {emergence_targets.shape}')
print(f'  Top8 samples: {len(top8_labels)} ({top8_labels.sum().int()} pos, '
      f'{(1 - top8_labels).sum().int()} neg)')

# 3-way temporal split
train_mask, val_mask, test_mask = _temporal_split(
    tourney_idx, data['tournament'].x.shape[0]
)
print(f'  Train: {train_mask.sum()}, Val: {val_mask.sum()}, Test: {test_mask.sum()}')

In [ ]:
# Move data to GPU
data = data.to(device)
emergence_targets = emergence_targets.to(device)
arch_idx = arch_idx.to(device)
tourney_idx = tourney_idx.to(device)
top8_labels = top8_labels.to(device)
train_mask = train_mask.to(device)
val_mask = val_mask.to(device)
test_mask = test_mask.to(device)

# Initialize model
node_dims = {nt: data[nt].x.shape[1] for nt in data.node_types}
model = MTGMetagameHGT(
    metadata=data.metadata(),
    node_dims=node_dims,
    hidden_dim=HIDDEN_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_HGT_LAYERS,
    dropout=DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
emergence_criterion = nn.MSELoss()
top8_criterion = nn.BCELoss()

In [ ]:
# Training loop
best_val_loss = float('inf')
best_epoch = 0
train_losses, val_losses, val_accs = [], [], []

start_time = time.time()
print(f'{"Epoch":>5} {"Train":>10} {"Emrg":>10} {"Top8":>10} {"Val":>10} {"Acc":>8} {"LR":>10}')
print('-' * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    output = model(data)
    node_emb = output['node_embeddings']

    emergence_loss = emergence_criterion(output['emergence_scores'], emergence_targets)
    top8_probs = model.predict_top8(
        node_emb['archetype'], node_emb['tournament'],
        arch_idx[train_mask], tourney_idx[train_mask],
    )
    top8_loss = top8_criterion(top8_probs, top8_labels[train_mask])
    total_loss = EMERGENCE_LOSS_WEIGHT * emergence_loss + TOP8_LOSS_WEIGHT * top8_loss

    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    train_losses.append(total_loss.item())

    if epoch % 5 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            val_output = model(data)
            val_emb = val_output['node_embeddings']
            val_top8_loss, val_acc, _, _, _ = _evaluate_top8(
                model, data, val_emb, arch_idx, tourney_idx,
                top8_labels, val_mask, top8_criterion,
            )
            val_emrg_loss = emergence_criterion(val_output['emergence_scores'], emergence_targets)
            val_loss = val_emrg_loss + val_top8_loss

        val_losses.append(val_loss.item())
        val_accs.append(val_acc.item())
        lr = scheduler.get_last_lr()[0]
        print(f'{epoch:5d} {total_loss.item():10.4f} {emergence_loss.item():10.4f} '
              f'{top8_loss.item():10.4f} {val_loss.item():10.4f} {val_acc.item():8.3f} '
              f'{lr:10.6f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss.item(),
                'val_acc': val_acc.item(),
            }, MODEL_PATH)

elapsed = time.time() - start_time
print(f'\nTraining complete in {elapsed:.1f}s ({elapsed/NUM_EPOCHS:.2f}s/epoch)')
print(f'Best model: epoch {best_epoch}, val_loss={best_val_loss:.4f}')

## 5. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(range(1, len(train_losses) + 1), train_losses, alpha=0.4, label='Train (per epoch)')
val_epochs = [1] + list(range(5, NUM_EPOCHS + 1, 5))
ax1.plot(val_epochs[:len(val_losses)], val_losses, 'o-', label='Validation', markersize=3)
ax1.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Validation accuracy
ax2.plot(val_epochs[:len(val_accs)], val_accs, 'o-', color='green', markersize=3)
ax2.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Random baseline')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Top-8 Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Final Evaluation (Held-Out Test Set)

In [ ]:
# Load best checkpoint
checkpoint = torch.load(MODEL_PATH, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

with torch.no_grad():
    output = model(data)
    node_emb = output['node_embeddings']

    # Emergence predictions
    print('Archetype Emergence Predictions:')
    print('=' * 40)
    scores = output['emergence_scores']
    arch_names = data['archetype'].names
    ranked = sorted(zip(arch_names, scores.cpu().tolist()), key=lambda x: -x[1])
    for name, score in ranked:
        arrow = '\u2191' if score > 0 else '\u2193'
        print(f'  {arrow} {name:25s} {score:+.4f}')

    # Test set evaluation
    print('\n\nTop 8 Prediction (TEST set - never seen during training):')
    print('=' * 55)
    test_loss, test_acc, test_prec, test_rec, test_f1 = _evaluate_top8(
        model, data, node_emb, arch_idx, tourney_idx,
        top8_labels, test_mask, top8_criterion,
    )
    print(f'  Accuracy:  {test_acc:.3f}')
    print(f'  Precision: {test_prec:.3f}')
    print(f'  Recall:    {test_rec:.3f}')
    print(f'  F1 Score:  {test_f1:.3f}')
    print(f'  BCE Loss:  {test_loss:.4f}')

    # Val set for comparison
    print('\n\nTop 8 Prediction (validation set - for comparison):')
    print('=' * 52)
    val_loss, val_acc, val_prec, val_rec, val_f1 = _evaluate_top8(
        model, data, node_emb, arch_idx, tourney_idx,
        top8_labels, val_mask, top8_criterion,
    )
    print(f'  Accuracy:  {val_acc:.3f}')
    print(f'  Precision: {val_prec:.3f}')
    print(f'  Recall:    {val_rec:.3f}')
    print(f'  F1 Score:  {val_f1:.3f}')
    print(f'  BCE Loss:  {val_loss:.4f}')

## 7. Generate Dashboard

In [ ]:
!python -m src.visualize

# Copy dashboard to Drive for easy access
import shutil
drive_path = '/content/drive/MyDrive/mtg-graph/mtg_dashboard.html'
if os.path.exists('/content/drive/MyDrive/mtg-graph'):
    shutil.copy('mtg_dashboard.html', drive_path)
    print(f'Dashboard copied to Google Drive: {drive_path}')